In [17]:
import math
import matplotlib.pyplot as plt
from numpy import arange, asarray, exp, array, float32
from numpy.random import normal
import numpy as np

from PySide6.QtWidgets import QApplication, QWidget, QPushButton, QMainWindow, QGridLayout, QFrame, QComboBox
from PySide6.QtCore import Slot, QSize, Signal, QObject
import pyqtgraph as pg
from pyqtgraph import PlotWidget

from pipython import GCSDevice, pitools
from pipython.pidevice.gcsmessages import GCSMessages
from pipython.pidevice.interfaces.piserial import PISerial
from pipython.pidevice.gcscommands import GCSCommands
from sys import platform
import time

from matplotlib.backend_bases import key_press_handler
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg, NavigationToolbar2Tk
from matplotlib.figure import Figure

In [18]:
def gaussian(x, sigma, mu):
    x = asarray(x)
    #y= (1/(sigma * math.sqrt(2 * math.pi )) * math.exp(-1/2 * (x-mu)**2 / sigma**2))
    return asarray(1/(sigma * math.sqrt(2 * math.pi )) * exp(-1/2 * (x-mu)**2 / sigma**2))

def define_x_grid(min, max, resolution):

    x = arange(min, max+resolution, resolution)

    return x

def create_random_gaussian(with_noise = False):
    random_x = np.random.Generator.random(-10, 10, size=None)
    y = gaussian(random_x, 1, 0)
    if with_noise == True:
        y += float(normal(scale=0.1, size=None))
    return random_x, y

def append_point(x, y, x_new, y_new):
    x.append(x_new)
    y.append(y_new)
    

In [38]:
class Data_contol(QObject):
    data_changed = Signal(np.ndarray, np.ndarray)

    def __init__(self, parent):
        super().__init__(parent)
        self.data = np.empty((2,0), dtype=float32)
        self.rng = np.random.default_rng()

    @Slot()
    def add_point(self, x, y):
        point = asarray([[x],[y]], dtype=float32)
        self.data = np.append(self.data, point, axis=1)
        self.data_changed.emit(self.data[0], self.data[1])

    def print_data(self):
        print(self.data)

    @Slot()
    def clear_data(self):
        self.data = np.empty((2,0), dtype=float32)
        self.data_changed.emit(self.data[0], self.data[1])

    def add_random_point(self):
        new_x = self.rng.uniform(low=-5, high=5)
        new_y = gaussian(new_x, 1.44, 0)
        self.add_point(new_x, new_y)
        self.data_changed.emit(self.data[0], self.data[1])

    @Slot()
    def add_noise(self):
        self.data[1,:] += self.rng.normal(scale=0.01, size=len(self.data[1,:]))
        self.data_changed.emit(self.data[0], self.data[1])


class Spectral_plot(PlotWidget):
    def __init__(self):
        super().__init__()
        self.plot_points = self.plot([],[], pen=None, symbol="o")# = Figure(dpi=75)

    #def plot_spectrum(self):
    #    plt.scatter(spectrum.data[0,:], spectrum.data[1,:])

    @Slot()
    def update_plot(self, data_x, data_y):
        self.plot_points.setData(data_x, data_y)

    #def clear_plot(self):
    #    self.update_plot(np.empty((2,0), dtype=float32))

#class Control_stage():
#    def __init__(self):
#       # if platform == "linux" or platform == "linux2":
#       #     self.port = '/dev/ttyS0',
#       # elif platform == "win32":
#       #     self.port='COM1'

#        self.pidevice = GCSDevice()
#        self.device = None
#        self.devices_list = ["test"]

#    @Slot()
#    def get_devices(self):
#        self.devices_list = list(self.pidevice.EnumerateTCPIPDevices(mask='C-884.4DB'))
#        if len(self.devices_list) != 0:
#            return self.devices_list
#        else:
#            return []
#    
#    def connect_device(self, device):
#        self.pidevice.ConnectTCPIPByDescription(device)
#
#    def print_identity(self):
#        self.pidevice.qIDN()
#
#    def move_stage_to_z(self, z_position):
#        pitools.moveandwait(self.pidevice, 'Axis_1', float32(z_position))

#class Control_stage_fake():
#    def __init__(self):
#        self.pidevice = "test"
#        self.pos = 0
#        self.device = None
#        self.devices_list = ["test"]
#
#    @Slot()
#    def get_devices(self):
#        self.devices_list = ["Device 1", "Device 2", "Device 3", "Device 4"]
#        if len(self.devices_list) != 0:
#            return self.devices_list
#        else:
#            return []
#    
#    def connect_device(self, device):
#        self.pidevice = device
#
#    def print_identity(self):
#        return self.pidevice
#
#    def move_stage_to_z(self, z_position):
#        self.pos = z_position
#        time.sleep(1)

class Plot_panel(QFrame):
    ################# Signals #################
    request_add_point = Signal()
    request_shutdown  = Signal()
    request_clear     = Signal()
    request_noise     = Signal()

    def __init__(self, parent):
        super().__init__(parent)
        plot_control_layout = QGridLayout()
        self.setLayout(plot_control_layout)

        ################# Buttons #################

        self.off_button = QPushButton("Off")
        self.off_button.clicked.connect(self.request_shutdown.emit)
        plot_control_layout.addWidget(self.off_button, 0, 0)

class Stage_panel(QFrame):
    request_device_list = Signal()

    def __init__(self, parent):
        super().__init__(parent)
        Stage_panel_layout = QGridLayout()
        self.setLayout(Stage_panel_layout)

        self.get_device_button = QPushButton("Get Devices")
        self.get_device_button.clicked.connect(self.request_device_list.emit)
        Stage_panel_layout.addWidget(self.get_device_button, 0, 0)

        self.select_device = QComboBox()
        Stage_panel_layout.addWidget(self.select_device, 0, 1)

    @Slot(list)
    def add_devices_to_menu(self, devices):
        self.select_device.addItems(devices)

class Data_panel(QFrame):
    request_reset_data = Signal()
    request_add_point = Signal()
    request_noise = Signal()

    def __init__(self, parent):
        super().__init__(parent)

        Data_panel_layout = QGridLayout()
        self.setLayout(Data_panel_layout)

        self.reset_data_button = QPushButton("Reset Data")
        self.reset_data_button.clicked.connect(self.request_reset_data.emit)
        Data_panel_layout.addWidget(self.reset_data_button, 0, 0)

        self.add_point_button =QPushButton("add")
        self.add_point_button.clicked.connect(self.request_add_point.emit)
        Data_panel_layout.addWidget(self.add_point_button, 1, 0)

        self.noise_button = QPushButton("Add Noise")
        self.noise_button.clicked.connect(self.request_noise.emit)
        Data_panel_layout.addWidget(self.noise_button, 2, 0)


class Stage_control_fake(QObject):
    devices_updated = Signal(list)

    def __init__(self, parent):
        super().__init__(parent)

        self.pidevice = "test"
        self.position = 0
        self.device = None
        self.devices_list = ["test"]

    @Slot()
    def get_devices(self):
        self.devices_list = ["Device 1", "Device 2", "Device 3", "Device 4"]
        if len(self.devices_list) != 0:
            self.devices_updated.emit(self.devices_list)
            #return self.devices_list
        else:
            self.devices_updated.emit(["No Devices Detected"])
            return []

    def connect_device(self, device):
        self.pidevice = device

    def print_identity(self):
        return self.pidevice

    def move_stage_to_z(self, z_position):
        self.pos = z_position
        time.sleep(1)

class Plot_control(QObject):    
    def __init__(self, parent):
        super().__init__(parent)

        # add something like refresh plot.

class Main_window(QMainWindow):
    def __init__(self):
        super().__init__()

        self.setWindowTitle("Data Aquisition")
        self.setMinimumSize(QSize(400,300))

        #self.data = data
        #self.stage = stage 

######################### UI #########################
# These implement buttons and the respective signals

        central_widget = QWidget()
        self.setCentralWidget(central_widget)

        central_layout = QGridLayout()
        central_widget.setLayout(central_layout)

        self.Plot_widget = Spectral_plot()
        central_layout.addWidget(self.Plot_widget, 0, 1, 1, 1) # row, column, rowspan, columspan

        self.Plot_panel = Plot_panel(self)  # User interaction with the plot
        central_layout.addWidget(self.Plot_panel, 1, 1, 1, 1) 

        self.Stage_panel = Stage_panel(self) # User interaction with the stage
        central_layout.addWidget(self.Stage_panel, 1, 0, 1, 1)

        self.Data_panel = Data_panel(self) # User interaction with the data
        central_layout.addWidget(self.Data_panel, 0, 0, 1, 1)





######################### Handlers #########################
# these accept signals and turn them into actions 

        self.Plot_controller = Plot_control(self) # turns user input to action 
        # for the plot this may not make sense, because the widget already implements a lot of this
        self.Stage_controller = Stage_control_fake(self) # turns user input to action and owns the current state of the stage

        self.Data_controller = Data_contol(self) # turns user input to action and owns the data



        ############################ Plot/Main Window Actions ############################ 

        self.Data_controller.data_changed.connect(self.Plot_widget.update_plot)
        #how to best control the main window?
        #eventually add a panel for app controll i guess. But for now thats not needed
        self.Plot_panel.request_shutdown.connect(self.shutdown)
        
        ############################ Stage Actions ############################ 
        self.Stage_panel.request_device_list.connect(self.Stage_controller.get_devices)
        self.Stage_controller.devices_updated.connect(self.Stage_panel.add_devices_to_menu)



        ############################ Data Actions ############################
        self.Data_panel.request_add_point.connect(self.Data_controller.add_random_point)
        self.Data_panel.request_reset_data.connect(self.Data_controller.clear_data)
        self.Data_panel.request_noise.connect(self.Data_controller.add_noise)

########################################################## Frame 2 ##########################################################   
#
#        self.z_stage_label = ttk.Label(self.frame2, text="Position Stage:")
#        self.z_stage_label.grid(row     = 0, 
#                                column  = 0,
#                                padx    = 5, 
#                                pady    = 5)
#
#        self.enter_z_stage = ttk.Entry(self.frame2)
#        self.enter_z_stage.grid(row     = 0, 
#                                column  = 1,
#                                padx    = 5, 
#                                pady    = 5, 
#                                sticky  = "ew")
#        
#        self.get_devices = ttk.Button(self.frame2, text="Get Devices", command=self.update_device_list)
#        self.get_devices.grid(row      = 1, 
#                               column   = 0,
#                               padx     = 5, 
#                               pady     = 5, 
#                               sticky   = "ew")
#
#
#        self.z_stage_select_controler = ttk.Combobox(self.frame2, state="readonly", textvariable=self.stage.device, values=self.stage.devices_list)
#        self.z_stage_select_controler.grid( row     = 1, 
#                                            column  = 1,
#                                            padx    = 5, 
#                                            pady    = 5, 
#                                            sticky  = "ew")

    @Slot()
    def shutdown(self):
        self.close()




In [39]:
#stage = Control_stage_fake()
#app.shutdown()
%gui qt
#gui.app.shutdown()



app = QApplication.instance()
#window = QWidget()
#data = Spectral_data()


window = Main_window()

window.show()
#gui.app.exec()

